> **Superseded (2026-09-22).** Use `python -m src.experiments.baseline_reproduction` (writes `reports/baseline_results_<dataset>.csv`). Kept for reference; its original outputs are in `archive/results_pre_fix_2026-09-22/`.

# 02 — Baseline Reproduction

**Purpose:** before building the adaptive resampler-selector (the novel
part of ARS-Stack), reproduce known baseline numbers from the literature
review using a **fixed** resampler (plain SMOTE) + standard models. This
gives two things:

1. Confirms the environment/pipeline works end-to-end.
2. Gives concrete numbers to beat later — if ARS-Stack doesn't clearly
   outperform these, that's a real (and reportable) finding, not a bug.

Reference numbers being targeted (from `configs/models/ensemble.yaml ->
baseline.reference_numbers`, same Credit Card Fraud dataset used in Papers
7, 9, 10):

| Source | Model | F1 | AUC-PR | AUC-ROC |
|---|---|---|---|---|
| Paper 7 | XGBoost + SMOTE | 0.89 | 0.93 | - |
| Paper 7 | EasyEnsemble | 0.87 | - | - |
| Paper 9 | Stacking + SMOTE | - | - | 0.943 |
| Paper 10 | Stacking + SMOTE-Tomek | 0.915 | 0.90 | - |

Note: exact numbers won't match perfectly since train/test splits and
feature preprocessing differ slightly from each paper's setup — the goal
is to land in the same range, not hit these to the decimal.


In [36]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import yaml
import numpy as np
import pandas as pd
import mlflow

from imblearn.over_sampling import SMOTE
from imblearn.metrics import geometric_mean_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, matthews_corrcoef
from xgboost import XGBClassifier

## Load configs

In [37]:
with open("../configs/base.yaml") as f:
    base_cfg = yaml.safe_load(f)

with open("../configs/models/ensemble.yaml") as f:
    model_cfg = yaml.safe_load(f)

baseline_cfg = model_cfg["baseline"]
print("Baseline resampler:", baseline_cfg["resampler"])
print("Models configured:", list(baseline_cfg["models"].keys()))


Baseline resampler: smote
Models configured: ['random_forest', 'xgboost', 'logistic_regression']


## Load processed data

Primary dataset for baseline comparison is `creditcard`, since that's the
exact dataset used in Papers 7, 9, and 10. Run
`python -m src.data.preprocess --dataset creditcard` first if this fails.

In [38]:
DATASET = "creditcard"
TARGET = "Class"

train_df = pd.read_csv(f"../data/processed/{DATASET}/{DATASET}_train.csv")
test_df = pd.read_csv(f"../data/processed/{DATASET}/{DATASET}_test.csv")

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train class distribution:\n", y_train.value_counts())


Train: (199364, 30), Test: (56962, 30)
Train class distribution:
 Class
0    199019
1       345
Name: count, dtype: int64


## Apply fixed SMOTE (baseline — no adaptive selection)

In [39]:
smote_cfg = baseline_cfg["smote"]
smote = SMOTE(k_neighbors=smote_cfg["k_neighbors"], random_state=smote_cfg["random_state"])
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_res).value_counts().to_dict())


Before SMOTE: {0: 199019, 1: 345}
After SMOTE:  {0: 199019, 1: 199019}


## Metric helper

Using F1, AUC-PR, G-mean, and MCC — not accuracy, since accuracy is misleading under imbalance (predicting all-majority already gives >99% accuracy on this dataset).

In [40]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        "f1": f1_score(y_true, y_pred),
        "auc_pr": average_precision_score(y_true, y_proba),
        "g_mean": geometric_mean_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

results = []  # collect dicts here, turned into a comparison table at the end


## MLflow setup

In [41]:
from mlflow.tracking import MlflowClient

# Initialize tracking URI and client
tracking_uri = base_cfg["project"]["mlflow_tracking_uri"]
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient(tracking_uri)

# Get the deleted experiment and restore it
exp = client.get_experiment_by_name(base_cfg["project"]["mlflow_experiment_name"])
if exp and exp.lifecycle_stage == "deleted":
    client.restore_experiment(exp.experiment_id)

# Now set_experiment will work without throwing an error
mlflow.set_experiment(base_cfg["project"]["mlflow_experiment_name"])

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1785776835695, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785778326338, lifecycle_stage='active', name='ars-stack-experiments', tags={}, trace_location=None, workspace='default'>

## Baseline 1: Random Forest + SMOTE

In [42]:
rf_cfg = baseline_cfg["models"]["random_forest"]

with mlflow.start_run(run_name="baseline_rf_smote"):
    mlflow.log_params({f"rf_{k}": v for k, v in rf_cfg.items()})
    mlflow.log_param("resampler", "smote")

    rf = RandomForestClassifier(**rf_cfg)
    rf.fit(X_train_res, y_train_res)

    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_proba)

    mlflow.log_metrics(metrics)
    results.append({"model": "RandomForest + SMOTE", **metrics})

print(metrics)


🏃 View run baseline_rf_smote at: http://localhost:5000/#/experiments/1/runs/c9f90cf33e25451691d6b2643adccbca
🧪 View experiment at: http://localhost:5000/#/experiments/1
{'f1': 0.8290155440414507, 'auc_pr': 0.867822095950217, 'g_mean': np.float64(0.9033887280985758), 'mcc': 0.8288259313544596}


## Baseline 2: XGBoost + SMOTE

This is the direct comparison point for Paper 7 (F1=0.89, AUC-PR=0.93).

In [43]:
xgb_cfg = baseline_cfg["models"]["xgboost"]

with mlflow.start_run(run_name="baseline_xgboost_smote"):
    mlflow.log_params({f"xgb_{k}": v for k, v in xgb_cfg.items()})
    mlflow.log_param("resampler", "smote")

    xgb = XGBClassifier(**xgb_cfg, use_label_encoder=False)
    xgb.fit(X_train_res, y_train_res)

    y_pred = xgb.predict(X_test)
    y_proba = xgb.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_proba)

    mlflow.log_metrics(metrics)
    results.append({"model": "XGBoost + SMOTE", **metrics})

print(metrics)


🏃 View run baseline_xgboost_smote at: http://localhost:5000/#/experiments/1/runs/bba1305db52f4d7483b3a60350a74210
🧪 View experiment at: http://localhost:5000/#/experiments/1
{'f1': 0.7887323943661971, 'auc_pr': 0.8728917115590162, 'g_mean': np.float64(0.925567705149038), 'mcc': 0.7908715406068779}


## Baseline 3: Simple Stacking (RF + XGBoost -> Logistic Regression)

Static stacking, no confidence weighting — this is the comparison point for Papers 9 and 10.

In [44]:
stack_cfg = baseline_cfg["simple_stacking"]
lr_cfg = baseline_cfg["models"]["logistic_regression"]

estimators = [
    ("random_forest", RandomForestClassifier(**rf_cfg)),
    ("xgboost", XGBClassifier(**xgb_cfg, use_label_encoder=False)),
]

with mlflow.start_run(run_name="baseline_simple_stacking_smote"):
    mlflow.log_param("resampler", "smote")
    mlflow.log_param("meta_learner", stack_cfg["meta_learner"])
    mlflow.log_param("cv_folds", stack_cfg["cv_folds"])

    stack = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(**lr_cfg),
        cv=stack_cfg["cv_folds"],
        passthrough=stack_cfg["passthrough"],
        n_jobs=-1,
    )
    stack.fit(X_train_res, y_train_res)

    y_pred = stack.predict(X_test)
    y_proba = stack.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_proba)

    mlflow.log_metrics(metrics)
    results.append({"model": "Simple Stacking (RF+XGB) + SMOTE", **metrics})

print(metrics)


0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.02s - Debugger warning: It seems that frozen modules are being used, which may
0

🏃 View run baseline_simple_stacking_smote at: http://localhost:5000/#/experiments/1/runs/7f53915ac6f74ec39f4e2bbb0e8a03a4
🧪 View experiment at: http://localhost:5000/#/experiments/1
{'f1': 0.82, 'auc_pr': 0.870608937880909, 'g_mean': np.float64(0.9145711566539908), 'mcc': 0.8198481341194773}


## Comparison Table: Your Results vs. Literature

In [45]:
results_df = pd.DataFrame(results).set_index("model")
results_df = results_df.round(4)

print("=== Your reproduced baseline results ===")
display(results_df)

print("\n=== Literature reference numbers (same dataset) ===")
ref = baseline_cfg["reference_numbers"]
ref_df = pd.DataFrame(ref).T
display(ref_df)

results_df.to_csv("../reports/baseline_results.csv")
print("\nSaved to ../reports/baseline_results.csv — these are the numbers ARS-Stack needs to beat.")


=== Your reproduced baseline results ===


,f1,auc_pr,g_mean,mcc
model,,,,
RandomForest + SMOTE,0.8290,0.8678,0.9034,0.8288
XGBoost + SMOTE,0.7887,0.8729,0.9256,0.7909
Simple Stacking (RF+XGB) + SMOTE,0.8200,0.8706,0.9146,0.8198



=== Literature reference numbers (same dataset) ===


,f1,auc_pr,auc_roc
paper_7_xgboost,0.890,0.93,NaN
paper_7_easyensemble,0.870,NaN,NaN
paper_9_stacking,NaN,NaN,0.943
paper_10_stacking_smote_tomek,0.915,0.90,NaN



Saved to ../reports/baseline_results.csv — these are the numbers ARS-Stack needs to beat.


## Takeaways (fill in after running)

- [ ] Environment/pipeline confirmed working end-to-end (data -> SMOTE -> model -> metrics -> MLflow)
- [ ] Reproduced F1/AUC-PR numbers are in the same ballpark as Papers 7/9/10 (don't need exact match)
- [ ] `reports/baseline_results.csv` saved — this is the baseline ARS-Stack must beat
- [ ] Noted which baseline (RF, XGBoost, or Simple Stacking) performs best here, as the strongest baseline to emphasize beating in the final report
- [ ] Confirmed MLflow is logging runs correctly (check `mlflow ui` at http://localhost:5000)
